# Baseline Model Development

This notebook trains and compares several baseline classification models for obesity-risk prediction.

## Objectives

- Recreate the stratified training, validation, and test datasets
- Use the reusable preprocessing module
- Establish a simple benchmark
- Train multiple classification algorithms
- Evaluate models using consistent metrics
- Select promising models for hyperparameter tuning
- Keep the test dataset untouched until final model selection

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [2]:
CURRENT_PATH = Path.cwd()

possible_roots = [
    CURRENT_PATH,
    *CURRENT_PATH.parents,
]

PROJECT_ROOT = next(
    (
        path
        for path in possible_roots
        if (path / "src" / "preprocessing.py").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root."
    )

project_root_string = str(PROJECT_ROOT)

if project_root_string not in sys.path:
    sys.path.insert(0, project_root_string)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "obesity.csv"
)

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATA_PATH.exists())

Project root: c:\Users\User\OneDrive\Desktop\Obesity-Risk-Intelligence-System
Dataset exists: True


In [3]:
from src.preprocessing import (
    PREDICTIVE_FEATURES,
    build_preprocessor,
)

print(
    "Configured predictive features:",
    len(PREDICTIVE_FEATURES),
)

test_preprocessor = build_preprocessor()

print(
    "Preprocessor type:",
    type(test_preprocessor).__name__,
)

Configured predictive features: 16
Preprocessor type: ColumnTransformer


In [4]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (20758, 18)


,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


In [5]:
IDENTIFIER_COLUMN = "id"
TARGET_COLUMN = "NObeyesdad"
RANDOM_STATE = 42

In [6]:
X = df[PREDICTIVE_FEATURES].copy()
y = df[TARGET_COLUMN].copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (20758, 16)
Target shape: (20758,)


In [7]:
missing_features = (
    set(PREDICTIVE_FEATURES)
    - set(df.columns)
)

unexpected_features = (
    set(X.columns)
    - set(PREDICTIVE_FEATURES)
)

print("Missing features:", missing_features)
print("Unexpected features:", unexpected_features)

Missing features: set()
Unexpected features: set()


In [8]:
assert not missing_features
assert not unexpected_features
assert list(X.columns) == PREDICTIVE_FEATURES

print("Feature configuration validation passed")

Feature configuration validation passed


In [9]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_validation, X_test, y_validation, y_test = (
    train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=y_temp,
    )
)

In [10]:
split_summary = pd.DataFrame(
    {
        "Dataset": [
            "Training",
            "Validation",
            "Test",
        ],
        "Records": [
            len(X_train),
            len(X_validation),
            len(X_test),
        ],
        "Percentage": [
            len(X_train) / len(X) * 100,
            len(X_validation) / len(X) * 100,
            len(X_test) / len(X) * 100,
        ],
    }
)

split_summary["Percentage"] = (
    split_summary["Percentage"].round(2)
)

split_summary

,Dataset,Records,Percentage
0,Training,14530,70.0
1,Validation,3114,15.0
2,Test,3114,15.0


In [11]:
train_validation_overlap = set(
    X_train.index
) & set(X_validation.index)

train_test_overlap = set(
    X_train.index
) & set(X_test.index)

validation_test_overlap = set(
    X_validation.index
) & set(X_test.index)

print(
    "Training-validation overlap:",
    len(train_validation_overlap),
)

print(
    "Training-test overlap:",
    len(train_test_overlap),
)

print(
    "Validation-test overlap:",
    len(validation_test_overlap),
)

Training-validation overlap: 0
Training-test overlap: 0
Validation-test overlap: 0


In [12]:
assert X_train.shape == (14530, 16)
assert X_validation.shape == (3114, 16)
assert X_test.shape == (3114, 16)

assert len(X_train) == len(y_train)
assert len(X_validation) == len(y_validation)
assert len(X_test) == len(y_test)

assert not train_validation_overlap
assert not train_test_overlap
assert not validation_test_overlap

assert (
    len(X_train)
    + len(X_validation)
    + len(X_test)
    == len(X)
)

print(
    "Baseline modelling dataset preparation passed"
)

Baseline modelling dataset preparation passed


In [13]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

In [14]:
CLASS_ORDER = [
    "Insufficient_Weight",
    "Normal_Weight",
    "Overweight_Level_I",
    "Overweight_Level_II",
    "Obesity_Type_I",
    "Obesity_Type_II",
    "Obesity_Type_III",
]

In [15]:
most_frequent_training_class = (
    y_train
    .value_counts()
    .idxmax()
)

most_frequent_class_count = (
    y_train
    .value_counts()
    .max()
)

most_frequent_class_percentage = (
    most_frequent_class_count
    / len(y_train)
    * 100
)

print(
    "Most frequent training class:",
    most_frequent_training_class,
)

print(
    "Training percentage:",
    f"{most_frequent_class_percentage:.2f}%",
)

Most frequent training class: Obesity_Type_III
Training percentage: 19.49%


In [16]:
dummy_classifier = DummyClassifier(
    strategy="most_frequent"
)

dummy_classifier

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None


In [17]:
dummy_classifier.fit(
    X_train,
    y_train,
)

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](7,)","[0.12,0.15,0.14,...,0.19,0.12,0.12]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[object](7,)","['Insufficient_Weight','Normal_Weight','Obesity_Type_I',..., 'Obesity_Type_III','Overweight_Level_I','Overweight_Level_II']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](16,)","['Age','Height','Weight',...,'SMOKE','SCC','MTRANS']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,7
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,16
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [18]:
dummy_validation_predictions = (
    dummy_classifier.predict(
        X_validation
    )
)

In [19]:
dummy_prediction_counts = (
    pd.Series(
        dummy_validation_predictions,
        name="Predicted Class",
    )
    .value_counts()
)

dummy_prediction_counts

Predicted Class
Obesity_Type_III    3114
Name: count, dtype: int64

In [20]:
print(
    "Unique predicted classes:",
    np.unique(
        dummy_validation_predictions
    ),
)

print(
    "Number of unique predictions:",
    len(
        np.unique(
            dummy_validation_predictions
        )
    ),
)

Unique predicted classes: ['Obesity_Type_III']
Number of unique predictions: 1


In [21]:
def calculate_classification_metrics(
    model_name,
    y_true,
    y_pred,
):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "Balanced Accuracy": (
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "Macro F1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "Weighted F1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
    }

In [22]:
dummy_result = calculate_classification_metrics(
    model_name="Dummy Classifier",
    y_true=y_validation,
    y_pred=dummy_validation_predictions,
)

model_results = [
    dummy_result
]

model_comparison_df = (
    pd.DataFrame(model_results)
    .set_index("Model")
    .round(4)
)

model_comparison_df

,Accuracy,Balanced Accuracy,Macro F1,Weighted F1
Model,,,,
Dummy Classifier,0.1949,0.1429,0.0466,0.0636


In [23]:
dummy_classification_report = (
    classification_report(
        y_validation,
        dummy_validation_predictions,
        labels=CLASS_ORDER,
        output_dict=True,
        zero_division=0,
    )
)

dummy_report_df = (
    pd.DataFrame(
        dummy_classification_report
    )
    .transpose()
    .round(4)
)

dummy_report_df

,precision,recall,f1-score,support
Insufficient_Weight,0.0000,0.0000,0.0000,379.0000
Normal_Weight,0.0000,0.0000,0.0000,463.0000
Overweight_Level_I,0.0000,0.0000,0.0000,364.0000
Overweight_Level_II,0.0000,0.0000,0.0000,378.0000
Obesity_Type_I,0.0000,0.0000,0.0000,436.0000
Obesity_Type_II,0.0000,0.0000,0.0000,487.0000
Obesity_Type_III,0.1949,1.0000,0.3263,607.0000
accuracy,0.1949,0.1949,0.1949,0.1949
macro avg,0.0278,0.1429,0.0466,3114.0000
weighted avg,0.0380,0.1949,0.0636,3114.0000


In [24]:
dummy_confusion_values = confusion_matrix(
    y_validation,
    dummy_validation_predictions,
    labels=CLASS_ORDER,
)

dummy_confusion_df = pd.DataFrame(
    dummy_confusion_values,
    index=[
        f"Actual: {class_name}"
        for class_name in CLASS_ORDER
    ],
    columns=[
        f"Predicted: {class_name}"
        for class_name in CLASS_ORDER
    ],
)

dummy_confusion_df

,Predicted: Insufficient_Weight,Predicted: Normal_Weight,Predicted: Overweight_Level_I,Predicted: Overweight_Level_II,Predicted: Obesity_Type_I,Predicted: Obesity_Type_II,Predicted: Obesity_Type_III
Actual: Insufficient_Weight,0,0,0,0,0,0,379
Actual: Normal_Weight,0,0,0,0,0,0,463
Actual: Overweight_Level_I,0,0,0,0,0,0,364
Actual: Overweight_Level_II,0,0,0,0,0,0,378
Actual: Obesity_Type_I,0,0,0,0,0,0,436
Actual: Obesity_Type_II,0,0,0,0,0,0,487
Actual: Obesity_Type_III,0,0,0,0,0,0,607


In [25]:
unique_dummy_predictions = np.unique(
    dummy_validation_predictions
)

assert len(dummy_validation_predictions) == len(
    y_validation
), "Prediction count does not match validation records."

assert len(unique_dummy_predictions) == 1, (
    "The most-frequent dummy classifier "
    "should predict only one class."
)

assert unique_dummy_predictions[0] == (
    most_frequent_training_class
), "Dummy prediction differs from the training majority class."

assert dummy_confusion_values.sum() == len(
    y_validation
), "Confusion matrix does not contain every validation record."

assert 0 <= dummy_result["Accuracy"] <= 1
assert 0 <= dummy_result["Balanced Accuracy"] <= 1
assert 0 <= dummy_result["Macro F1"] <= 1
assert 0 <= dummy_result["Weighted F1"] <= 1

print("Dummy classifier validation passed")

Dummy classifier validation passed


In [26]:
logistic_regression_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            build_preprocessor(),
        ),
        (
            "classifier",
            LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                max_iter=1000,
            ),
        ),
    ]
)

logistic_regression_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: floa

In [27]:
logistic_regression_pipeline.fit(
    X_train,
    y_train,
)

c:\Users\User\OneDrive\Desktop\Obesity-Risk-Intelligence-System\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](7,)","['Insufficient_Weight','Normal_Weight','Obesity_Type_I',..., 'Obesity_Type_III','Overweight_Level_I','Overweight_Level_II']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](16,)","['Age','Height','Weight',...,'SMOKE','SCC','MTRANS']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,16
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the o

In [28]:
logistic_classifier = (
    logistic_regression_pipeline
    .named_steps["classifier"]
)

iterations_used = int(
    logistic_classifier.n_iter_.max()
)

print(
    "Maximum training iterations used:",
    iterations_used,
)

print(
    "Configured iteration limit:",
    logistic_classifier.max_iter,
)

print(
    "Converged before limit:",
    iterations_used
    < logistic_classifier.max_iter,
)

Maximum training iterations used: 285
Configured iteration limit: 1000
Converged before limit: True


In [29]:
logistic_validation_predictions = (
    logistic_regression_pipeline.predict(
        X_validation
    )
)

In [30]:
logistic_prediction_counts = (
    pd.Series(
        logistic_validation_predictions,
        name="Predicted Class",
    )
    .value_counts()
    .reindex(CLASS_ORDER, fill_value=0)
)

logistic_prediction_counts

Predicted Class
Insufficient_Weight    412
Normal_Weight          433
Overweight_Level_I     345
Overweight_Level_II    366
Obesity_Type_I         440
Obesity_Type_II        511
Obesity_Type_III       607
Name: count, dtype: int64

In [31]:
logistic_unique_predictions = np.unique(
    logistic_validation_predictions
)

print(
    "Number of predicted classes:",
    len(logistic_unique_predictions),
)

print(
    "Predicted classes:",
    logistic_unique_predictions,
)

Number of predicted classes: 7
Predicted classes: ['Insufficient_Weight' 'Normal_Weight' 'Obesity_Type_I' 'Obesity_Type_II'
 'Obesity_Type_III' 'Overweight_Level_I' 'Overweight_Level_II']


In [32]:
logistic_result = calculate_classification_metrics(
    model_name="Logistic Regression",
    y_true=y_validation,
    y_pred=logistic_validation_predictions,
)

logistic_result

{'Model': 'Logistic Regression',
 'Accuracy': 0.8580603725112396,
 'Balanced Accuracy': 0.8432419109973445,
 'Macro F1': 0.8418091057474137,
 'Weighted F1': 0.8565990669044098}

In [33]:
model_results.append(
    logistic_result
)

model_comparison_df = (
    pd.DataFrame(model_results)
    .set_index("Model")
    .sort_values(
        by="Macro F1",
        ascending=False,
    )
    .round(4)
)

model_comparison_df

,Accuracy,Balanced Accuracy,Macro F1,Weighted F1
Model,,,,
Logistic Regression,0.8581,0.8432,0.8418,0.8566
Dummy Classifier,0.1949,0.1429,0.0466,0.0636


In [34]:
logistic_improvement = {
    "Accuracy improvement": (
        logistic_result["Accuracy"]
        - dummy_result["Accuracy"]
    ),
    "Balanced accuracy improvement": (
        logistic_result["Balanced Accuracy"]
        - dummy_result["Balanced Accuracy"]
    ),
    "Macro F1 improvement": (
        logistic_result["Macro F1"]
        - dummy_result["Macro F1"]
    ),
    "Weighted F1 improvement": (
        logistic_result["Weighted F1"]
        - dummy_result["Weighted F1"]
    ),
}

logistic_improvement_df = pd.Series(
    logistic_improvement,
    name="Improvement",
).round(4)

logistic_improvement_df

Accuracy improvement             0.6631
Balanced accuracy improvement    0.7004
Macro F1 improvement             0.7952
Weighted F1 improvement          0.7930
Name: Improvement, dtype: float64

In [35]:
logistic_classification_report = (
    classification_report(
        y_validation,
        logistic_validation_predictions,
        labels=CLASS_ORDER,
        output_dict=True,
        zero_division=0,
    )
)

logistic_report_df = (
    pd.DataFrame(
        logistic_classification_report
    )
    .transpose()
    .round(4)
)

logistic_report_df

,precision,recall,f1-score,support
Insufficient_Weight,0.8689,0.9446,0.9052,379.0000
Normal_Weight,0.8591,0.8035,0.8304,463.0000
Overweight_Level_I,0.7275,0.6896,0.7080,364.0000
Overweight_Level_II,0.7022,0.6799,0.6909,378.0000
Obesity_Type_I,0.8159,0.8234,0.8196,436.0000
Obesity_Type_II,0.9198,0.9651,0.9419,487.0000
Obesity_Type_III,0.9967,0.9967,0.9967,607.0000
accuracy,0.8581,0.8581,0.8581,0.8581
macro avg,0.8415,0.8432,0.8418,3114.0000
weighted avg,0.8561,0.8581,0.8566,3114.0000


In [36]:
logistic_class_metrics = (
    logistic_report_df
    .loc[CLASS_ORDER]
    .copy()
)

strongest_logistic_class = (
    logistic_class_metrics["f1-score"]
    .idxmax()
)

weakest_logistic_class = (
    logistic_class_metrics["f1-score"]
    .idxmin()
)

print(
    "Strongest class by F1-score:",
    strongest_logistic_class,
)

print(
    "Strongest class F1-score:",
    logistic_class_metrics.loc[
        strongest_logistic_class,
        "f1-score",
    ],
)

print(
    "Weakest class by F1-score:",
    weakest_logistic_class,
)

print(
    "Weakest class F1-score:",
    logistic_class_metrics.loc[
        weakest_logistic_class,
        "f1-score",
    ],
)

Strongest class by F1-score: Obesity_Type_III
Strongest class F1-score: 0.9967
Weakest class by F1-score: Overweight_Level_II
Weakest class F1-score: 0.6909


In [37]:
logistic_confusion_values = confusion_matrix(
    y_validation,
    logistic_validation_predictions,
    labels=CLASS_ORDER,
)

logistic_confusion_df = pd.DataFrame(
    logistic_confusion_values,
    index=[
        f"Actual: {class_name}"
        for class_name in CLASS_ORDER
    ],
    columns=[
        f"Predicted: {class_name}"
        for class_name in CLASS_ORDER
    ],
)

logistic_confusion_df

,Predicted: Insufficient_Weight,Predicted: Normal_Weight,Predicted: Overweight_Level_I,Predicted: Overweight_Level_II,Predicted: Obesity_Type_I,Predicted: Obesity_Type_II,Predicted: Obesity_Type_III
Actual: Insufficient_Weight,358,20,1,0,0,0,0
Actual: Normal_Weight,52,372,32,6,1,0,0
Actual: Overweight_Level_I,1,36,251,69,7,0,0
Actual: Overweight_Level_II,0,4,51,257,58,7,1
Actual: Obesity_Type_I,1,1,8,33,359,33,1
Actual: Obesity_Type_II,0,0,1,1,15,470,0
Actual: Obesity_Type_III,0,0,1,0,0,1,605


In [38]:
logistic_correct_predictions = np.trace(
    logistic_confusion_values
)

logistic_incorrect_predictions = (
    len(y_validation)
    - logistic_correct_predictions
)

print(
    "Correct validation predictions:",
    logistic_correct_predictions,
)

print(
    "Incorrect validation predictions:",
    logistic_incorrect_predictions,
)

print(
    "Total validation predictions:",
    len(y_validation),
)

Correct validation predictions: 2672
Incorrect validation predictions: 442
Total validation predictions: 3114


In [39]:
fitted_logistic_preprocessor = (
    logistic_regression_pipeline
    .named_steps["preprocessor"]
)

logistic_feature_names = (
    fitted_logistic_preprocessor
    .get_feature_names_out()
)

logistic_coefficients = (
    logistic_classifier.coef_
)

print(
    "Number of transformed features:",
    len(logistic_feature_names),
)

print(
    "Coefficient matrix shape:",
    logistic_coefficients.shape,
)

print(
    "Model classes:",
    logistic_classifier.classes_,
)

Number of transformed features: 25
Coefficient matrix shape: (7, 25)
Model classes: ['Insufficient_Weight' 'Normal_Weight' 'Obesity_Type_I' 'Obesity_Type_II'
 'Obesity_Type_III' 'Overweight_Level_I' 'Overweight_Level_II']


In [40]:
assert len(
    logistic_validation_predictions
) == len(y_validation)

assert len(
    logistic_unique_predictions
) > 1

assert logistic_confusion_values.shape == (
    len(CLASS_ORDER),
    len(CLASS_ORDER),
)

assert logistic_confusion_values.sum() == len(
    y_validation
)

assert (
    logistic_correct_predictions
    + logistic_incorrect_predictions
    == len(y_validation)
)

assert logistic_coefficients.shape == (
    len(logistic_classifier.classes_),
    len(logistic_feature_names),
)

for metric_name in [
    "Accuracy",
    "Balanced Accuracy",
    "Macro F1",
    "Weighted F1",
]:
    assert 0 <= logistic_result[
        metric_name
    ] <= 1

assert logistic_result["Accuracy"] > (
    dummy_result["Accuracy"]
)

assert logistic_result["Macro F1"] > (
    dummy_result["Macro F1"]
)

print(
    "Logistic Regression validation passed"
)

Logistic Regression validation passed


In [41]:
decision_tree_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            build_preprocessor(),
        ),
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

decision_tree_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: floa

In [42]:
decision_tree_pipeline.fit(
    X_train,
    y_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](7,)","['Insufficient_Weight','Normal_Weight','Obesity_Type_I',..., 'Obesity_Type_III','Overweight_Level_I','Overweight_Level_II']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](16,)","['Age','Height','Weight',...,'SMOKE','SCC','MTRANS']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,16
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the o

In [43]:
decision_tree_training_predictions = (
    decision_tree_pipeline.predict(
        X_train
    )
)

decision_tree_validation_predictions = (
    decision_tree_pipeline.predict(
        X_validation
    )
)

In [44]:
decision_tree_result = (
    calculate_classification_metrics(
        model_name="Decision Tree",
        y_true=y_validation,
        y_pred=decision_tree_validation_predictions,
    )
)

decision_tree_result

{'Model': 'Decision Tree',
 'Accuracy': 0.8452151573538856,
 'Balanced Accuracy': 0.8311678589584047,
 'Macro F1': 0.8308013528086666,
 'Weighted F1': 0.8456445626719233}

In [45]:
decision_tree_training_result = (
    calculate_classification_metrics(
        model_name="Decision Tree Training",
        y_true=y_train,
        y_pred=decision_tree_training_predictions,
    )
)

decision_tree_training_result

{'Model': 'Decision Tree Training',
 'Accuracy': 1.0,
 'Balanced Accuracy': 1.0,
 'Macro F1': 1.0,
 'Weighted F1': 1.0}

In [46]:
decision_tree_generalisation_df = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Balanced Accuracy",
            "Macro F1",
            "Weighted F1",
        ],
        "Training": [
            decision_tree_training_result["Accuracy"],
            decision_tree_training_result[
                "Balanced Accuracy"
            ],
            decision_tree_training_result["Macro F1"],
            decision_tree_training_result[
                "Weighted F1"
            ],
        ],
        "Validation": [
            decision_tree_result["Accuracy"],
            decision_tree_result[
                "Balanced Accuracy"
            ],
            decision_tree_result["Macro F1"],
            decision_tree_result[
                "Weighted F1"
            ],
        ],
    }
)

decision_tree_generalisation_df[
    "Difference"
] = (
    decision_tree_generalisation_df["Training"]
    - decision_tree_generalisation_df["Validation"]
)

decision_tree_generalisation_df.round(4)

,Metric,Training,Validation,Difference
0,Accuracy,1.0,0.8452,0.1548
1,Balanced Accuracy,1.0,0.8312,0.1688
2,Macro F1,1.0,0.8308,0.1692
3,Weighted F1,1.0,0.8456,0.1544


In [47]:
model_results.append(
    decision_tree_result
)

model_comparison_df = (
    pd.DataFrame(model_results)
    .set_index("Model")
    .sort_values(
        by="Macro F1",
        ascending=False,
    )
    .round(4)
)

model_comparison_df

,Accuracy,Balanced Accuracy,Macro F1,Weighted F1
Model,,,,
Logistic Regression,0.8581,0.8432,0.8418,0.8566
Decision Tree,0.8452,0.8312,0.8308,0.8456
Dummy Classifier,0.1949,0.1429,0.0466,0.0636


In [48]:
decision_tree_prediction_counts = (
    pd.Series(
        decision_tree_validation_predictions,
        name="Predicted Class",
    )
    .value_counts()
    .reindex(
        CLASS_ORDER,
        fill_value=0,
    )
)

decision_tree_prediction_counts

Predicted Class
Insufficient_Weight    382
Normal_Weight          444
Overweight_Level_I     373
Overweight_Level_II    391
Obesity_Type_I         438
Obesity_Type_II        480
Obesity_Type_III       606
Name: count, dtype: int64

In [49]:
decision_tree_unique_predictions = np.unique(
    decision_tree_validation_predictions
)

print(
    "Number of predicted classes:",
    len(decision_tree_unique_predictions),
)

print(
    "Predicted classes:",
    decision_tree_unique_predictions,
)

Number of predicted classes: 7
Predicted classes: ['Insufficient_Weight' 'Normal_Weight' 'Obesity_Type_I' 'Obesity_Type_II'
 'Obesity_Type_III' 'Overweight_Level_I' 'Overweight_Level_II']


In [50]:
decision_tree_classification_report = (
    classification_report(
        y_validation,
        decision_tree_validation_predictions,
        labels=CLASS_ORDER,
        output_dict=True,
        zero_division=0,
    )
)

decision_tree_report_df = (
    pd.DataFrame(
        decision_tree_classification_report
    )
    .transpose()
    .round(4)
)

decision_tree_report_df

,precision,recall,f1-score,support
Insufficient_Weight,0.8717,0.8786,0.8752,379.0000
Normal_Weight,0.8086,0.7754,0.7916,463.0000
Overweight_Level_I,0.6756,0.6923,0.6839,364.0000
Overweight_Level_II,0.7187,0.7434,0.7308,378.0000
Obesity_Type_I,0.8037,0.8073,0.8055,436.0000
Obesity_Type_II,0.9396,0.9261,0.9328,487.0000
Obesity_Type_III,0.9967,0.9951,0.9959,607.0000
accuracy,0.8452,0.8452,0.8452,0.8452
macro avg,0.8306,0.8312,0.8308,3114.0000
weighted avg,0.8463,0.8452,0.8456,3114.0000


In [51]:
decision_tree_class_metrics = (
    decision_tree_report_df
    .loc[CLASS_ORDER]
    .copy()
)

strongest_decision_tree_class = (
    decision_tree_class_metrics["f1-score"]
    .idxmax()
)

weakest_decision_tree_class = (
    decision_tree_class_metrics["f1-score"]
    .idxmin()
)

print(
    "Strongest class:",
    strongest_decision_tree_class,
)

print(
    "Strongest class F1-score:",
    decision_tree_class_metrics.loc[
        strongest_decision_tree_class,
        "f1-score",
    ],
)

print(
    "Weakest class:",
    weakest_decision_tree_class,
)

print(
    "Weakest class F1-score:",
    decision_tree_class_metrics.loc[
        weakest_decision_tree_class,
        "f1-score",
    ],
)

Strongest class: Obesity_Type_III
Strongest class F1-score: 0.9959
Weakest class: Overweight_Level_I
Weakest class F1-score: 0.6839


In [52]:
decision_tree_confusion_values = confusion_matrix(
    y_validation,
    decision_tree_validation_predictions,
    labels=CLASS_ORDER,
)

decision_tree_confusion_df = pd.DataFrame(
    decision_tree_confusion_values,
    index=[
        f"Actual: {class_name}"
        for class_name in CLASS_ORDER
    ],
    columns=[
        f"Predicted: {class_name}"
        for class_name in CLASS_ORDER
    ],
)

decision_tree_confusion_df

,Predicted: Insufficient_Weight,Predicted: Normal_Weight,Predicted: Overweight_Level_I,Predicted: Overweight_Level_II,Predicted: Obesity_Type_I,Predicted: Obesity_Type_II,Predicted: Obesity_Type_III
Actual: Insufficient_Weight,333,40,5,1,0,0,0
Actual: Normal_Weight,47,359,46,8,3,0,0
Actual: Overweight_Level_I,1,37,252,58,16,0,0
Actual: Overweight_Level_II,0,6,51,281,33,7,0
Actual: Obesity_Type_I,1,2,18,41,352,21,1
Actual: Obesity_Type_II,0,0,1,2,32,451,1
Actual: Obesity_Type_III,0,0,0,0,2,1,604


In [53]:
fitted_decision_tree = (
    decision_tree_pipeline
    .named_steps["classifier"]
)

print(
    "Tree depth:",
    fitted_decision_tree.get_depth(),
)

print(
    "Number of leaf nodes:",
    fitted_decision_tree.get_n_leaves(),
)

print(
    "Number of model classes:",
    len(fitted_decision_tree.classes_),
)

Tree depth: 27
Number of leaf nodes: 1794
Number of model classes: 7


In [54]:
decision_tree_preprocessor = (
    decision_tree_pipeline
    .named_steps["preprocessor"]
)

decision_tree_feature_names = (
    decision_tree_preprocessor
    .get_feature_names_out()
)

decision_tree_importances = (
    fitted_decision_tree.feature_importances_
)

print(
    "Transformed feature count:",
    len(decision_tree_feature_names),
)

print(
    "Feature importance count:",
    len(decision_tree_importances),
)

print(
    "Total feature importance:",
    decision_tree_importances.sum(),
)

Transformed feature count: 25
Feature importance count: 25
Total feature importance: 0.9999999999999999


In [55]:
decision_tree_importance_df = (
    pd.DataFrame(
        {
            "Feature": decision_tree_feature_names,
            "Importance": decision_tree_importances,
        }
    )
    .sort_values(
        by="Importance",
        ascending=False,
    )
    .reset_index(drop=True)
)

decision_tree_importance_df.head(10)

,Feature,Importance
0,numerical__Weight,0.445344
1,nominal__Gender_Male,0.193295
2,numerical__Height,0.141011
3,numerical__Age,0.069195
4,numerical__CH2O,0.026218
5,numerical__FCVC,0.019395
6,numerical__FAF,0.016957
7,numerical__NCP,0.016767
8,numerical__TUE,0.016400
9,ordinal__CALC,0.012279


In [56]:
decision_tree_vs_dummy = (
    decision_tree_result["Macro F1"]
    - dummy_result["Macro F1"]
)

decision_tree_vs_logistic = (
    decision_tree_result["Macro F1"]
    - logistic_result["Macro F1"]
)

print(
    "Macro F1 improvement over Dummy:",
    round(decision_tree_vs_dummy, 4),
)

print(
    "Macro F1 difference from Logistic Regression:",
    round(decision_tree_vs_logistic, 4),
)

Macro F1 improvement over Dummy: 0.7842
Macro F1 difference from Logistic Regression: -0.011


In [57]:
assert len(
    decision_tree_training_predictions
) == len(y_train)

assert len(
    decision_tree_validation_predictions
) == len(y_validation)

assert len(
    decision_tree_unique_predictions
) > 1

assert decision_tree_confusion_values.shape == (
    len(CLASS_ORDER),
    len(CLASS_ORDER),
)

assert decision_tree_confusion_values.sum() == len(
    y_validation
)

assert len(
    decision_tree_feature_names
) == len(
    decision_tree_importances
)

assert np.isclose(
    decision_tree_importances.sum(),
    1.0,
)

for metric_name in [
    "Accuracy",
    "Balanced Accuracy",
    "Macro F1",
    "Weighted F1",
]:
    assert 0 <= decision_tree_result[
        metric_name
    ] <= 1

assert decision_tree_result["Accuracy"] > (
    dummy_result["Accuracy"]
)

assert decision_tree_result["Macro F1"] > (
    dummy_result["Macro F1"]
)

print(
    "Decision Tree validation passed"
)

Decision Tree validation passed


In [58]:
random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            build_preprocessor(),
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

random_forest_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: floa

In [59]:
random_forest_pipeline.fit(
    X_train,
    y_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](7,)","['Insufficient_Weight','Normal_Weight','Obesity_Type_I',..., 'Obesity_Type_III','Overweight_Level_I','Overweight_Level_II']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](16,)","['Age','Height','Weight',...,'SMOKE','SCC','MTRANS']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,16
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the o

In [60]:
random_forest_training_predictions = (
    random_forest_pipeline.predict(
        X_train
    )
)

random_forest_validation_predictions = (
    random_forest_pipeline.predict(
        X_validation
    )
)

In [61]:
random_forest_result = (
    calculate_classification_metrics(
        model_name="Random Forest",
        y_true=y_validation,
        y_pred=random_forest_validation_predictions,
    )
)

random_forest_result

{'Model': 'Random Forest',
 'Accuracy': 0.8956326268464997,
 'Balanced Accuracy': 0.8845427653140858,
 'Macro F1': 0.8848580437360324,
 'Weighted F1': 0.8952538368641886}

In [62]:
random_forest_training_result = (
    calculate_classification_metrics(
        model_name="Random Forest Training",
        y_true=y_train,
        y_pred=random_forest_training_predictions,
    )
)

random_forest_training_result

{'Model': 'Random Forest Training',
 'Accuracy': 1.0,
 'Balanced Accuracy': 1.0,
 'Macro F1': 1.0,
 'Weighted F1': 1.0}

In [63]:
random_forest_generalisation_df = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Balanced Accuracy",
            "Macro F1",
            "Weighted F1",
        ],
        "Training": [
            random_forest_training_result[
                "Accuracy"
            ],
            random_forest_training_result[
                "Balanced Accuracy"
            ],
            random_forest_training_result[
                "Macro F1"
            ],
            random_forest_training_result[
                "Weighted F1"
            ],
        ],
        "Validation": [
            random_forest_result[
                "Accuracy"
            ],
            random_forest_result[
                "Balanced Accuracy"
            ],
            random_forest_result[
                "Macro F1"
            ],
            random_forest_result[
                "Weighted F1"
            ],
        ],
    }
)

random_forest_generalisation_df[
    "Difference"
] = (
    random_forest_generalisation_df["Training"]
    - random_forest_generalisation_df["Validation"]
)

random_forest_generalisation_df.round(4)

,Metric,Training,Validation,Difference
0,Accuracy,1.0,0.8956,0.1044
1,Balanced Accuracy,1.0,0.8845,0.1155
2,Macro F1,1.0,0.8849,0.1151
3,Weighted F1,1.0,0.8953,0.1047


In [64]:
model_results.append(
    random_forest_result
)

model_comparison_df = (
    pd.DataFrame(model_results)
    .set_index("Model")
    .sort_values(
        by="Macro F1",
        ascending=False,
    )
    .round(4)
)

model_comparison_df

,Accuracy,Balanced Accuracy,Macro F1,Weighted F1
Model,,,,
Random Forest,0.8956,0.8845,0.8849,0.8953
Logistic Regression,0.8581,0.8432,0.8418,0.8566
Decision Tree,0.8452,0.8312,0.8308,0.8456
Dummy Classifier,0.1949,0.1429,0.0466,0.0636


In [65]:
random_forest_prediction_counts = (
    pd.Series(
        random_forest_validation_predictions,
        name="Predicted Class",
    )
    .value_counts()
    .reindex(
        CLASS_ORDER,
        fill_value=0,
    )
)

random_forest_prediction_counts

Predicted Class
Insufficient_Weight    380
Normal_Weight          480
Overweight_Level_I     341
Overweight_Level_II    381
Obesity_Type_I         432
Obesity_Type_II        494
Obesity_Type_III       606
Name: count, dtype: int64

In [66]:
random_forest_unique_predictions = np.unique(
    random_forest_validation_predictions
)

print(
    "Number of predicted classes:",
    len(random_forest_unique_predictions),
)

print(
    "Predicted classes:",
    random_forest_unique_predictions,
)

Number of predicted classes: 7
Predicted classes: ['Insufficient_Weight' 'Normal_Weight' 'Obesity_Type_I' 'Obesity_Type_II'
 'Obesity_Type_III' 'Overweight_Level_I' 'Overweight_Level_II']


In [67]:
random_forest_classification_report = (
    classification_report(
        y_validation,
        random_forest_validation_predictions,
        labels=CLASS_ORDER,
        output_dict=True,
        zero_division=0,
    )
)

random_forest_report_df = (
    pd.DataFrame(
        random_forest_classification_report
    )
    .transpose()
    .round(4)
)

random_forest_report_df

,precision,recall,f1-score,support
Insufficient_Weight,0.9184,0.9208,0.9196,379.0000
Normal_Weight,0.8458,0.8769,0.8611,463.0000
Overweight_Level_I,0.8006,0.7500,0.7745,364.0000
Overweight_Level_II,0.8084,0.8148,0.8116,378.0000
Obesity_Type_I,0.8796,0.8716,0.8756,436.0000
Obesity_Type_II,0.9474,0.9610,0.9541,487.0000
Obesity_Type_III,0.9983,0.9967,0.9975,607.0000
accuracy,0.8956,0.8956,0.8956,0.8956
macro avg,0.8855,0.8845,0.8849,3114.0000
weighted avg,0.8952,0.8956,0.8953,3114.0000


In [68]:
random_forest_class_metrics = (
    random_forest_report_df
    .loc[CLASS_ORDER]
    .copy()
)

strongest_random_forest_class = (
    random_forest_class_metrics[
        "f1-score"
    ]
    .idxmax()
)

weakest_random_forest_class = (
    random_forest_class_metrics[
        "f1-score"
    ]
    .idxmin()
)

print(
    "Strongest class:",
    strongest_random_forest_class,
)

print(
    "Strongest class F1-score:",
    random_forest_class_metrics.loc[
        strongest_random_forest_class,
        "f1-score",
    ],
)

print(
    "Weakest class:",
    weakest_random_forest_class,
)

print(
    "Weakest class F1-score:",
    random_forest_class_metrics.loc[
        weakest_random_forest_class,
        "f1-score",
    ],
)

Strongest class: Obesity_Type_III
Strongest class F1-score: 0.9975
Weakest class: Overweight_Level_I
Weakest class F1-score: 0.7745


In [69]:
random_forest_confusion_values = confusion_matrix(
    y_validation,
    random_forest_validation_predictions,
    labels=CLASS_ORDER,
)

random_forest_confusion_df = pd.DataFrame(
    random_forest_confusion_values,
    index=[
        f"Actual: {class_name}"
        for class_name in CLASS_ORDER
    ],
    columns=[
        f"Predicted: {class_name}"
        for class_name in CLASS_ORDER
    ],
)

random_forest_confusion_df

,Predicted: Insufficient_Weight,Predicted: Normal_Weight,Predicted: Overweight_Level_I,Predicted: Overweight_Level_II,Predicted: Obesity_Type_I,Predicted: Obesity_Type_II,Predicted: Obesity_Type_III
Actual: Insufficient_Weight,349,29,1,0,0,0,0
Actual: Normal_Weight,29,406,25,3,0,0,0
Actual: Overweight_Level_I,1,37,273,44,9,0,0
Actual: Overweight_Level_II,0,7,34,308,25,4,0
Actual: Obesity_Type_I,1,1,7,25,380,21,1
Actual: Obesity_Type_II,0,0,0,1,18,468,0
Actual: Obesity_Type_III,0,0,1,0,0,1,605


In [70]:
random_forest_correct_predictions = np.trace(
    random_forest_confusion_values
)

random_forest_incorrect_predictions = (
    len(y_validation)
    - random_forest_correct_predictions
)

print(
    "Correct validation predictions:",
    random_forest_correct_predictions,
)

print(
    "Incorrect validation predictions:",
    random_forest_incorrect_predictions,
)

print(
    "Total validation predictions:",
    len(y_validation),
)

Correct validation predictions: 2789
Incorrect validation predictions: 325
Total validation predictions: 3114


In [71]:
fitted_random_forest = (
    random_forest_pipeline
    .named_steps["classifier"]
)

print(
    "Number of trained trees:",
    len(fitted_random_forest.estimators_),
)

print(
    "Number of classes:",
    len(fitted_random_forest.classes_),
)

print(
    "Configured number of trees:",
    fitted_random_forest.n_estimators,
)

Number of trained trees: 100
Number of classes: 7
Configured number of trees: 100


In [72]:
random_forest_tree_depths = [
    tree.get_depth()
    for tree in fitted_random_forest.estimators_
]

random_forest_leaf_counts = [
    tree.get_n_leaves()
    for tree in fitted_random_forest.estimators_
]

print(
    "Average tree depth:",
    round(
        np.mean(random_forest_tree_depths),
        2,
    ),
)

print(
    "Minimum tree depth:",
    min(random_forest_tree_depths),
)

print(
    "Maximum tree depth:",
    max(random_forest_tree_depths),
)

print(
    "Average leaf count:",
    round(
        np.mean(random_forest_leaf_counts),
        2,
    ),
)

Average tree depth: 27.59
Minimum tree depth: 23
Maximum tree depth: 36
Average leaf count: 2246.46


In [73]:
random_forest_preprocessor = (
    random_forest_pipeline
    .named_steps["preprocessor"]
)

random_forest_feature_names = (
    random_forest_preprocessor
    .get_feature_names_out()
)

random_forest_importances = (
    fitted_random_forest.feature_importances_
)

In [74]:
random_forest_importance_df = (
    pd.DataFrame(
        {
            "Feature": random_forest_feature_names,
            "Importance": random_forest_importances,
        }
    )
    .sort_values(
        by="Importance",
        ascending=False,
    )
    .reset_index(drop=True)
)

random_forest_importance_df.head(10)

,Feature,Importance
0,numerical__Weight,0.367761
1,numerical__FCVC,0.093627
2,numerical__Height,0.091364
3,numerical__Age,0.088868
4,nominal__Gender_Female,0.048035
5,numerical__CH2O,0.041533
6,numerical__TUE,0.041036
7,numerical__FAF,0.037309
8,nominal__Gender_Male,0.035519
9,numerical__NCP,0.030870


In [75]:
print(
    "Transformed feature count:",
    len(random_forest_feature_names),
)

print(
    "Feature importance count:",
    len(random_forest_importances),
)

print(
    "Total feature importance:",
    random_forest_importances.sum(),
)

Transformed feature count: 25
Feature importance count: 25
Total feature importance: 1.0


In [76]:
random_forest_vs_dummy = (
    random_forest_result["Macro F1"]
    - dummy_result["Macro F1"]
)

random_forest_vs_logistic = (
    random_forest_result["Macro F1"]
    - logistic_result["Macro F1"]
)

random_forest_vs_decision_tree = (
    random_forest_result["Macro F1"]
    - decision_tree_result["Macro F1"]
)

print(
    "Macro F1 improvement over Dummy:",
    round(random_forest_vs_dummy, 4),
)

print(
    "Macro F1 difference from Logistic Regression:",
    round(random_forest_vs_logistic, 4),
)

print(
    "Macro F1 difference from Decision Tree:",
    round(random_forest_vs_decision_tree, 4),
)

Macro F1 improvement over Dummy: 0.8382
Macro F1 difference from Logistic Regression: 0.043
Macro F1 difference from Decision Tree: 0.0541


In [77]:
decision_tree_macro_f1_gap = (
    decision_tree_training_result["Macro F1"]
    - decision_tree_result["Macro F1"]
)

random_forest_macro_f1_gap = (
    random_forest_training_result["Macro F1"]
    - random_forest_result["Macro F1"]
)

overfitting_comparison_df = pd.DataFrame(
    {
        "Model": [
            "Decision Tree",
            "Random Forest",
        ],
        "Training Macro F1": [
            decision_tree_training_result[
                "Macro F1"
            ],
            random_forest_training_result[
                "Macro F1"
            ],
        ],
        "Validation Macro F1": [
            decision_tree_result[
                "Macro F1"
            ],
            random_forest_result[
                "Macro F1"
            ],
        ],
        "Difference": [
            decision_tree_macro_f1_gap,
            random_forest_macro_f1_gap,
        ],
    }
)

overfitting_comparison_df.round(4)

,Model,Training Macro F1,Validation Macro F1,Difference
0,Decision Tree,1.0,0.8308,0.1692
1,Random Forest,1.0,0.8849,0.1151


In [78]:
assert len(
    random_forest_training_predictions
) == len(y_train)

assert len(
    random_forest_validation_predictions
) == len(y_validation)

assert len(
    random_forest_unique_predictions
) > 1

assert len(
    fitted_random_forest.estimators_
) == fitted_random_forest.n_estimators

assert random_forest_confusion_values.shape == (
    len(CLASS_ORDER),
    len(CLASS_ORDER),
)

assert random_forest_confusion_values.sum() == len(
    y_validation
)

assert (
    random_forest_correct_predictions
    + random_forest_incorrect_predictions
    == len(y_validation)
)

assert len(
    random_forest_feature_names
) == len(
    random_forest_importances
)

assert np.isclose(
    random_forest_importances.sum(),
    1.0,
)

for metric_name in [
    "Accuracy",
    "Balanced Accuracy",
    "Macro F1",
    "Weighted F1",
]:
    assert 0 <= random_forest_result[
        metric_name
    ] <= 1

assert random_forest_result["Accuracy"] > (
    dummy_result["Accuracy"]
)

assert random_forest_result["Macro F1"] > (
    dummy_result["Macro F1"]
)

print("Random Forest validation passed")

Random Forest validation passed
